# Tables and storage

Start from ordinary tables, create an AnnNet graph, and compare
storage/interchange formats on the same small graph.


In [ ]:
import annnet as an


## Create a graph from a table


In [ ]:
import polars as pl

edges = pl.DataFrame(
    {
        'edge_id': ['e1', 'e1_parallel', 'e2', 'e3'],
        'source': ['A', 'A', 'B', 'C'],
        'target': ['B', 'B', 'C', 'D'],
        'weight': [0.9, 0.45, 0.7, 0.4],
        'directed': [True, True, True, False],
        'relation': ['activates', 'phosphorylates', 'inhibits', 'binds'],
        'slice': ['baseline', 'stimulated', 'stimulated', 'stimulated'],
    }
)

G = an.from_dataframe(edges, schema='edge_list')
print('shape:', G.shape)
print('slices:', G.slices.list())


## Export dataframe tables


In [ ]:
tables = an.to_dataframes(G)
print('tables:', sorted(tables))
tables['edges'].select(['edge_id', 'source', 'target', 'relation', 'weight'])


## Visualize the imported edge table

This confirms that the table was interpreted as the intended
directed and undirected interactions.


In [ ]:
from annnet.utils import plotting

plotting.plot(
    G,
    backend='graphviz',
    show_edge_labels=True,
    edge_label_keys=['relation'],
)


## CSV and Excel ingestion


In [ ]:
import os
import tempfile
import pandas as pd

tmp = tempfile.mkdtemp(prefix='annnet_tables_')
csv_path = os.path.join(tmp, 'edges.csv')
xlsx_path = os.path.join(tmp, 'edges.xlsx')

edges.write_csv(csv_path)
pd.DataFrame(edges.to_dicts()).to_excel(xlsx_path, index=False)

from_csv = an.from_csv(csv_path)
from_excel = an.from_excel(xlsx_path)

print('csv shape:', from_csv.shape)
print('excel shape:', from_excel.shape)
print()
print(open(csv_path, encoding='utf-8').read())


## Native, Parquet, JSON, and NDJSON outputs


In [ ]:
native_path = os.path.join(tmp, 'graph.annnet')
parquet_path = os.path.join(tmp, 'graph_parquet')
json_path = os.path.join(tmp, 'graph.json')
ndjson_path = os.path.join(tmp, 'graph_ndjson')

an.write(G, native_path)
an.to_parquet(G, parquet_path)
an.to_json(G, json_path, indent=2)
an.write_ndjson(G, ndjson_path)

round_trips = {
    'native': an.read(native_path).shape,
    'parquet': an.from_parquet(parquet_path).shape,
    'json': an.from_json(json_path).shape,
    'ndjson files': sorted(os.listdir(ndjson_path)),
}
round_trips


## Inspect a JSON and NDJSON payload


In [ ]:
import json

with open(json_path, encoding='utf-8') as handle:
    json_doc = json.load(handle)
with open(os.path.join(ndjson_path, 'edges.ndjson'), encoding='utf-8') as handle:
    edge_lines = [line.strip() for line in handle if line.strip()]

print('JSON top-level keys:', sorted(json_doc))
print('JSON edges:')
print(json.dumps(json_doc['edges'], indent=2))
print('edges.ndjson:')
for line in edge_lines:
    print(json.dumps(json.loads(line), indent=2))


CSV and Excel are useful at human-facing boundaries. Native
`.annnet` and Parquet are better when AnnNet remains the source
of record. JSON and NDJSON are convenient when another system
expects plain structured text.
